In [74]:
import pandas as pd
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Common imports
import numpy as np
import os
import matplotlib as mpl
mpl.use('TkAgg') 
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
plt.ion()

In [75]:
def load_AllOfTheData():
    df = pd.read_csv('../Parsing/CSVDatawithDT.txt')
    
    return df

In [76]:
data = load_AllOfTheData()

data.head()

,'ThreatId','SourceSystem','ThreatName','Priority','Description','Affiliation','Location','Location.Latitude','Location.Longitude','Location.Altitude',...,'Tracks.Lob.OriginPosition.Lla.Altitude','Tracks.Lob.OriginPosition.Ecef','Tracks.Lob.OriginPosition.DataCase','Tracks.Feature.Importance.Importance','Countermeasures.DeviceId','Countermeasures.Countermeasure','Countermeasures.State','Tracks.Feature.Countermeasures.DroneModel','Countermeasures.DroneModel','DetectionTimeInDT'
0,3784,TPM,DON 3784,8,NaN,2,NaN,42.491578,-71.311929,1263.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07-13 15:08:02.495332
1,3786,TPM,DON 3786,8,NaN,2,NaN,42.414375,-71.351943,10382.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07-13 15:07:59.591863
2,3726,TPM,DON 3726,8,NaN,2,NaN,42.770994,-71.986791,5065.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07-13 15:08:02.233450
3,3794,TPM,DON 3794,8,NaN,2,NaN,42.287262,-71.336343,3855.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07-13 15:08:01.878965
4,3797,TPM,DON 3797,8,NaN,2,NaN,42.321132,-71.247895,4228.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07-13 15:08:02.521955


In [77]:
data.columns = data.columns.str.strip() #There's extra white space in the data
data.columns = data.columns.str.replace("'", "") #The columns already contain quotes we need to remove 
#data.fillna('0', inplace=True)

ThreatId = data['ThreatId'].to_numpy()
Latitude = data['Location.Latitude'].to_numpy()
Longitude = data['Location.Longitude'].to_numpy()
Altitude = data['Location.Altitude'].to_numpy()
DetectionTime = data['DetectionTimeInDT'].to_numpy()


In [78]:
# create a dictionary and store each ThreatID in it, 
# if the Threat is already in the dictionary, append the current L,L,A coordinates
# we then can iterate through Dictionary and plot each threat ID

In [79]:
ThreatsXY = {}
ThreatsHT = {}

for i, threat_num in enumerate(ThreatId):


    latitude = Latitude[i]
    longitude = Longitude[i]
    altitude = Altitude[i]
    detection_time = DetectionTime[i] # string value 


    new_coordinate_xy = [latitude, longitude, detection_time] # first graph 
    new_coordinate_ht = [altitude, detection_time] # second graph 


    # populating dictionary for longitude vs latitude graph 
    # eliminating duplicates in dictionary without using a series  
    if threat_num in ThreatsXY: 
        if (new_coordinate_xy not in ThreatsXY[threat_num]):
            ThreatsXY[threat_num].append(new_coordinate_xy)
    
    else:
        ThreatsXY[threat_num] = [new_coordinate_xy]


    # populating dictionary for height vs time graph 
    # eliminating duplicates in dictionary without using a series  
    if threat_num in ThreatsHT: 
        if new_coordinate_ht[0] == 0.0:
            continue
        elif new_coordinate_ht not in ThreatsHT[threat_num]:
            ThreatsHT[threat_num].append(new_coordinate_ht)


    else: 
        ThreatsHT[threat_num] = [new_coordinate_ht]


# print both dictionarys 
print("XY Threats",  len(ThreatsXY[4069]))
print("HT Threats",  len(ThreatsHT[4069]))


XY Threats 156
HT Threats 144


In [80]:
threat_counts = data['ThreatId'].value_counts()

print(threat_counts)

ThreatId
3709    677
1006    677
3633    535
3801    489
3746    423
       ... 
4079      1
4096      1
3832      1
4147      1
4100      1
Name: count, Length: 226, dtype: int64


In [81]:
# h t graph 

# for the current id 
for id, coord in list(ThreatsHT.items())[0:4]: 
    graph_data = {"Height":[], "Time":[], "ThreatID":[]} 
    num_points = 0
    for i in range(len(coord)):
        graph_data['Height'].append(coord[i][0])
        graph_data['Time'].append(coord[i][1])

        num_points +=1
    
    # plt.figure(figsize=(15,8))
    # plt.plot(graph_data['Time'], graph_data['Height'], label=f"Number of hits: {num_points}" , linestyle='-', marker='x')
    # plt.xlabel('Time')
    # plt.xticks(rotation=45, ha='right')
    # plt.ylabel('Height')
    # plt.title(f'Flight Time for ID: {id}')
    # plt.legend()

    






In [82]:

# x y graph 

# for the current id 
for id, coord in list(ThreatsXY.items())[4:8]:
    graph_data = {"Longitude":[], "Latitude":[], "ThreatID":[]}
    for i in range(len(ThreatsXY[id])):
        # append coordinates to the dictionary lists
        graph_data["Longitude"].append(coord[i][0])
        graph_data["Latitude"].append(coord[i][1])

    # # plot scatter data
    # plt.figure(figsize=(10,6))
    # plt.title(f'Top level view of {id}')
    # plt.xlabel('Longitude')
    # plt.ylabel('Latitude')
    # plt.plot(graph_data["Latitude"], graph_data["Longitude"], linestyle='-', marker='x')


In [83]:
Detection_times = {}

# Fill detection_times dictionary to show,  {id: array_of_time_change_in_seconds}
for i, threat in enumerate(ThreatId):
    id = ThreatId[i]
    date = DetectionTime[i]
    
    date_split = date.split(':')
    seconds = date_split[2]
    if id not in Detection_times:
        Detection_times[id] = [float(seconds)]
    else:
        Detection_times[id].append(float(seconds))
#create new feature if Signal is persistent (detected frequently within 6 seconds)

data['Signal Persistence'] = False
signal_pers = []
#check if there is an time when the diff is less than or equal to 6 seconds, if so change feature to True
for id, seconds in Detection_times.items():
    for left in range(len(seconds)-5):
        right = left + 5
        diff = seconds[right] - seconds[left]
        if diff <= 6.0:
            if id not in signal_pers:
                signal_pers.append(id)
            data.loc[data['ThreatId'] == id, "Signal Persistence"] = True
            break

#print(len(signal_pers))

In [84]:
Detection_heights = {}

for i, threat in enumerate(ThreatId):
    id = ThreatId[i]
    altitude = Altitude[i]

    if id not in Detection_heights:
        Detection_heights[id] = [float(altitude)]
    else:
        Detection_heights[id].append(float(altitude))

data['Altitude Consistent'] = False

altitude_const = []
for id, heights in Detection_heights.items():
    for left in range(len(heights)-5):
        right = left +5
        if diff <= 6.0:
            if id not in altitude_const:
                altitude_const.append(id)
            data.loc[data['ThreatId'] == id, "Altitude Consistent"] = True
            break
#print(len(altitude_const))
            


In [85]:
#update speed to mph
data["Speed"] = data["Speed"] * 2.237



In [86]:
#verifying ninja ID's match excel sheet
ninja_ids = []

data["Drone"] = False

for id in ThreatId:
    ninja_status = data.loc[data["ThreatId"] == id, "Ninja"]
    if any(ninja_status) == True:
        data.loc[data["ThreatId"] == id, "Drone"] = True
        if id not in ninja_ids:
            ninja_ids.append(id)

In [87]:
# # Generate Latitude vs. Longitude, Height vs. Time, Slope vs. Time plots for one specific ID

# IDofInterest = 4069

from datetime import datetime
import numpy as np
from scipy.interpolate import BSpline
from scipy.interpolate import splrep, splev,CubicSpline
import matplotlib.pyplot as plt

# # x y graph 

# graph_data_x = {"Longitude":[], "Latitude":[], "ThreatID":[]}
# for i in range(len(ThreatsXY[IDofInterest])):
#     # append coordinates to the dictionary lists
#     graph_data_x["Longitude"].append(ThreatsXY[IDofInterest][i][0])
#     graph_data_x["Latitude"].append(ThreatsXY[IDofInterest][i][1])

# # plot scatter data
# # plt.figure(figsize=(10,6))
# # plt.title(f'Top level view of {IDofInterest}')
# # plt.xlabel('Longitude')
# # plt.ylabel('Latitude')
# # plt.plot(graph_data["Latitude"], graph_data["Longitude"], linestyle='-', marker='x')

# # h t Graph

# graph_data = {"Height":[], "DeltaT":[], "Slope":[], "Detection Time":[]}
# t0 = datetime.strptime(ThreatsHT[IDofInterest][0][1],"%Y-%m-%d %H:%M:%S.%f")
# for i in range(len(ThreatsHT[IDofInterest])):
#     # append coordinates to the dictionary lists
#     ti = datetime.strptime(ThreatsHT[IDofInterest][i][1],"%Y-%m-%d %H:%M:%S.%f")
#     #print(t0)

#     time_diff = ti - t0
#     deltat = time_diff.total_seconds()

#     if deltat not in graph_data["DeltaT"]:
#         graph_data["DeltaT"].append(deltat)
#         graph_data["Height"].append(ThreatsHT[IDofInterest][i][0])
#     else:
#         next

# # plot scatter data
# # plt.figure(figsize=(10,6))
# # plt.title(f'Top level view of {IDofInterest}')
# # plt.xlabel('Longitude')
# # plt.ylabel('Latitude')
# # plt.plot(graph_data["Latitude"], graph_data["Longitude"], linestyle='-', marker='x')

# # slope t graph

# SlopeROC = []

# for i in range(0,len(graph_data["Height"])-1):

#     point1 = [graph_data["Height"][i], graph_data["DeltaT"][i]]
#     point2 = [graph_data["Height"][i+1], graph_data["DeltaT"][i+1]]

#     if point2[0] != point1[0]:
#         m = (point2[1] - point1[1])/(point2[0]-point1[0])
#         new_slope = [m, graph_data["DeltaT"][i]]
#     else:
#         new_slope = [0, graph_data["DeltaT"][i]]

#     SlopeROC.append(new_slope)


# #plotting the slope changes for the scatter data
# for i in range(len(SlopeROC)):
#     # append coordinates to the dictionary lists
    
#     graph_data["Slope"].append(SlopeROC[i][0])
#     graph_data["Detection Time"].append(SlopeROC[i][1])


# #plot scatter data for slope changes
# # plt.figure(figsize=(20,8))
# # plt.title(f'Slopes of {IDofInterest}')
# # plt.xlabel('Detection Time (seconds from T0)')
# # plt.ylabel('Slope')
# # plt.scatter(graph_data["Detection Time"], graph_data["Slope"], marker='.') 
 
# # adding the spline to the slope plot

# x = graph_data["Detection Time"]
# y = graph_data["Slope"]

# # Find spline representation
# tck = CubicSpline(x, y)

# # Evaluate spline at new points
# xnew = np.linspace(0, max(x), 1000)
# ynew = tck(xnew)

# # Plotting
# plt.figure(figsize=(20,8))
# plt.plot(xnew, ynew, c='r')
# plt.scatter(x, y, c='g')
# plt.title(f'Spline Interpolation of Slopes of {IDofInterest}')
# plt.xlabel('Time Since First Detection (seconds)')
# plt.ylabel('S(x)')
# plt.show()

In [88]:
# # Trajectory B-Spline of one ID of Interest

# IDofInterest = 4144

# from datetime import datetime

# # Height Time Graph

# graph_data = {"Height":[], "DeltaT":[]}
# t0 = datetime.strptime(ThreatsHT[IDofInterest][0][1],"%Y-%m-%d %H:%M:%S.%f")
# for i in range(len(ThreatsHT[IDofInterest])):
#     # append coordinates to the dictionary lists
#     try:
#         ti = datetime.strptime(ThreatsHT[IDofInterest][i][1],"%Y-%m-%d %H:%M:%S.%f")
#     except ValueError:
#         next

#     time_diff = ti - t0
#     deltat = time_diff.total_seconds()

#     if deltat not in graph_data["DeltaT"]:
#         graph_data["DeltaT"].append(deltat)
#         graph_data["Height"].append(ThreatsHT[IDofInterest][i][0])
#     else:
#         next

# # Sorting the time values in case they are not already sorted (happens for a couple tracks)
# HTCoords = []
# for i in range(0,len(graph_data["DeltaT"])):
#     HTCoords.append([graph_data["DeltaT"][i],graph_data["Height"][i]])

# sorted_coords = sorted(HTCoords)

# graph_data = {"Height":[], "DeltaT":[]}
# for i in range(0, len(sorted_coords)):
#     graph_data["DeltaT"].append(sorted_coords[i][0])
#     graph_data["Height"].append(sorted_coords[i][1])

# #plot scatter data for slope changes
# # plt.figure(figsize=(20,8))
# # plt.title(f'Height vs. Time of {IDofInterest}')
# # plt.xlabel('Time Since First Detection (seconds)')
# # plt.ylabel('Height')
# # plt.scatter(graph_data["DeltaT"], graph_data["Height"], marker='.') 

# # finding the b-spline of the height vs. time trajectory

# x = graph_data["DeltaT"]
# y = graph_data["Height"]

# # Find spline representation
# tck = splrep(x, y, k=2)

# # Evaluate spline at new points
# xnew = np.linspace(0, max(x), 500)
# ynew = splev(xnew, tck)

# # Plotting
# plt.figure(figsize=(20,8))
# plt.scatter(x, y, c='b', marker='.', zorder=2)
# plt.plot(xnew, ynew, c='r', linestyle='-', zorder=1)
# plt.title(f'Spline Interpolation of Trajectory of {IDofInterest}')
# plt.xlabel('Time Since First Detection (seconds)')
# plt.ylabel('S(x)')
# plt.show()

In [89]:
# Getting a dictionary of the spline coords for each drone

SplinedTrajectories = {}

IDsofInterest = ninja_ids

from datetime import datetime

for IDofInterest in IDsofInterest:

    # Height Time Graph
    graph_data = {"Height":[], "DeltaT":[]}
    t0 = datetime.strptime(ThreatsHT[IDofInterest][0][1],"%Y-%m-%d %H:%M:%S.%f")
    for i in range(len(ThreatsHT[IDofInterest])):
        # append coordinates to the dictionary lists
        try:
            ti = datetime.strptime(ThreatsHT[IDofInterest][i][1],"%Y-%m-%d %H:%M:%S.%f")
        except ValueError:
            next
        #print(t0)

        time_diff = ti - t0
        deltat = time_diff.total_seconds()

        if deltat not in graph_data["DeltaT"]:
            graph_data["DeltaT"].append(deltat)
            graph_data["Height"].append(ThreatsHT[IDofInterest][i][0])
        else:
            next

    #plot scatter data for slope changes
    # plt.figure(figsize=(20,8))
    # plt.title(f'Height vs. Time of {IDofInterest}')
    # plt.xlabel('Time Since First Detection (seconds)')
    # plt.ylabel('Height')
    # plt.scatter(graph_data["DeltaT"], graph_data["Height"], marker='.', c='b') 

     # corner case where the times are not in order already
    HTCoords = []
    for i in range(0,len(graph_data["DeltaT"])):
        HTCoords.append([graph_data["DeltaT"][i],graph_data["Height"][i]])
    
    sorted_coords = sorted(HTCoords)

    graph_data = {"Height":[], "DeltaT":[]}
    for i in range(0, len(sorted_coords)):
        graph_data["DeltaT"].append(sorted_coords[i][0])
        graph_data["Height"].append(sorted_coords[i][1])

    # Find spline representation of each point's height vs. time graph

    x = graph_data["DeltaT"]
    y = graph_data["Height"]

    # tck = splrep(x, y,s=5)
    tck = splrep(x, y)

    # Evaluate spline at new points
    xnew = np.linspace(0, max(x), 2000)
    # ynew = splev(xnew, tck)
    ynew = splev(xnew,tck)

    # Plotting
    # plt.figure(figsize=(20,8))
    # plt.scatter(x, y, c='b', marker='.', zorder=2)
    # # plt.plot(xnew, ynew, c='r', linestyle='-', zorder=1)
    # plt.plot(xnew, ynew, c='m', linestyle='-', zorder=1)
    # plt.title(f'Spline Interpolation of Trajectory of {IDofInterest}')
    # plt.xlabel('Time Since First Detection (seconds)')
    # plt.ylabel('S(x): Height (meters)')
    # plt.show()

    SplinedTrajectories[IDofInterest] = [list(xnew),list(ynew)]

# SplinedTrajectories is a dictionary containing all of the drone ids and each drone id is a key that references two lists: at SplinedTrajectories[DroneID][0] are the time coords, at SplinedTrajectories[DroneID][1] are the height coords


In [90]:
X = data[["Ninja", "Signal Persistence", "Altitude Consistent", "Speed"]]
y = data["Drone"]

In [91]:
import sklearn
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [92]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = DecisionTreeClassifier(random_state=42)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

In [93]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

object_columns = data.select_dtypes(include=['object']).columns

for col in object_columns:
    data[col] = label_encoder.fit_transform(data[col])


correlation = data.corr(numeric_only=False)["Drone"].sort_values(ascending=False)
print(correlation)

Drone                                   1.000000
SignalConfidence                        0.831696
Ninja                                   0.785105
SourceTypes.IsNinjaThreat               0.785105
Affiliation                             0.740191
                                          ...   
Tracks.Lob.OriginPosition.Lla                NaN
Tracks.Lob.OriginPosition.Ecef               NaN
Tracks.Feature.Importance.Importance         NaN
Countermeasures.Countermeasure               NaN
Countermeasures.State                        NaN
Name: Drone, Length: 119, dtype: float64


In [94]:
X = data.drop(columns=["Drone", "ThreatId", "SourceSystem", "ThreatName", "Tracks.Lob.OriginPosition.DataCase", "Countermeasures.State"])
y = data["Drone"]

X = X.dropna(axis=1, how='all')

print(X.shape)

(12801, 71)


In [95]:
import sklearn
import xgboost as xgb
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRFClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import make_scorer, r2_score
from sklearn.feature_selection import SelectKBest, f_classif,chi2
from sklearn.impute import SimpleImputer



In [96]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

selector = SelectKBest(f_classif, k=20)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)


models = {
    "dt_clf": DecisionTreeClassifier(random_state=42),
    "xgb_clas" : XGBRFClassifier(),
    "rnd_forest": RandomForestClassifier()

}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    r2 = r2_score(y_pred,y_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name}: Accuracy = {accuracy}")
    print(f"{name}: R2 score = {r2}")


c:\Users\umeny\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: UserWarning: Features [ 9 10 18 19 29 40 41 48 52 58 59] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\umeny\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:113: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


dt_clf: Accuracy = 0.9918000780944943
dt_clf: R2 score = 0.9390175868285879
xgb_clas: Accuracy = 0.9539242483404919
xgb_clas: R2 score = 0.6194938302694535
rnd_forest: Accuracy = 0.9949238578680203
rnd_forest: R2 score = 0.9619478129678946


In [97]:
# from sklearn.tree import plot_tree
# import matplotlib.pyplot as plt

# feature_names = ["Ninja", "Signal Persistence", "Altitude Consistent", "Speed"]

# plt.figure(figsize=(12,8))
# plot_tree(clf, feature_names= feature_names, class_names= ["False", "True"], filled=True)
# plt.show()

clf = RandomForestClassifier(random_state=42)

clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)


In [98]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = cross_val_score(clf, X, y, cv=kf, scoring=make_scorer(r2_score))

print("Cross-Validation R2 Scores for each fold:", r2_scores)
print("Mean R2 Score:", r2_scores.mean())
print("Standard Deviation of R2 Score:", r2_scores.std())


Cross-Validation R2 Scores for each fold: [0.96232332 0.96406296 0.97502937 0.95912737 0.95261911]
Mean R2 Score: 0.962632425490073
Standard Deviation of R2 Score: 0.007326365089612851


In [99]:
target_id = 3797

row = data[data["ThreatId"] == target_id]

current_row = row.drop(columns=["Drone", "ThreatId", "SourceSystem", "ThreatName",  "Tracks.Lob.OriginPosition.DataCase", "Countermeasures.State"])

current_row = current_row[X_train.columns]

current_row_imputed = imputer.transform(current_row)

current_row_scaled = scaler.transform(current_row_imputed)

prediction = clf.predict(current_row_scaled)

print(f"The prediction for {target_id}: is {prediction[0]}")



The prediction for 3797: is True


In [116]:
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Impute, scale, and fit KMeans on X
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

kmeans = KMeans(n_clusters=2, random_state=42)
kmeans.fit(X_scaled)

# Predict the cluster for a specific target ID
target_id = 3801
current_row = data[data["ThreatId"] == target_id].drop(
    columns=["Drone", "ThreatId", "SourceSystem", "ThreatName"]
)
current_row = current_row[X.columns]  # Match columns to X

current_row_imputed = imputer.transform(current_row)
current_row_scaled = scaler.transform(current_row_imputed)

predicted_cluster = kmeans.predict(current_row_scaled)

# Print distances to centroids
distances = kmeans.transform(current_row_scaled)

# Add cluster assignments to the DataFrame
data['Cluster'] = kmeans.labels_

# Analyze clusters
cluster_summary = data.groupby('Cluster')['Drone'].mean()
most_likely_true_cluster = cluster_summary.idxmax()

print(f"Cluster {most_likely_true_cluster} has the highest proportion of True Drone values.")

# IDs in the most likely True cluster
likely_true_ids = data[data['Cluster'] == most_likely_true_cluster]['ThreatId']
print(f"IDs in the most likely True cluster (Cluster {most_likely_true_cluster}): {likely_true_ids.tolist()}")


Cluster 1 has the highest proportion of True Drone values.
IDs in the most likely True cluster (Cluster 1): [3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 3801, 4007, 4007, 4007, 4007, 4007, 4007, 4007, 4007, 4007, 4007, 4007, 4007, 4007, 4007, 4007, 4007, 4057, 4057, 4057, 4057, 4057, 4057, 4057, 4057, 4057, 4049, 4049, 4049, 4049, 4049, 4098, 4098, 4098, 4098, 4098, 4098, 4098, 4106, 4106, 4106, 4106, 4106, 4106, 4106, 4106, 4106, 4106, 4106, 4116, 4116, 4116, 4116, 4116, 4116, 4116, 4116, 4116, 4116, 4129, 4129, 4129, 4129, 4129, 4129, 4129, 4129, 4129]
